In [11]:

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from datasets import load_from_disk
from qwen_vl_utils import process_vision_info
import os
import torch
import torch.nn.functional as F

import dotenv
dotenv.load_dotenv()

True

In [12]:

REASONING_MODEL = 'Jakh0103/Qwen2.5-VL-3B-GRPO-VSR'

ORIGINAL_DATASET_PATH =  os.environ['DATA_PATH'] + "/vsr"
OUTPUT_DATASET_PATH =  os.environ['DATA_PATH'] + "/vsr_prompt_tuning"

# default: Load the model on the available device(s)
reasoning_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    REASONING_MODEL, torch_dtype="auto", device_map="cuda:0"
)

# default processor
processor = AutoProcessor.from_pretrained(REASONING_MODEL, use_fast=True)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:

# preprocess the dataset
dataset = load_from_disk(ORIGINAL_DATASET_PATH)['train']

# TODO Remove
dataset = dataset.select(range(0, 35))

dataset = dataset.map(lambda sample: {
    "problem": f'Is the following statement true: {sample["caption"]}',
    "solution": str(sample["label"]==1)
}, remove_columns=["caption", "label", "relation", "subj", "obj"], desc="Preprocessing dataset")



def make_conversation_from_prompt(example, prompt_template, answer = None):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{example['image_path']}"},
                {"type": "text", "text": prompt_template(example)},
            ],
        }
    ]
    if answer is not None:
        messages.append(
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer},
                ],
            }
        )
    return {"messages": messages, "solution": example['solution']}

# apply formatting
def reasoning_prompt_template(example):
    return f"{example['problem']} First output the thinking process in <think> </think> tags and then output the final answer in <answer> </answer> tags."

def baseline_prompt_template(example):
    return f"{example['problem']} First output the thinking process in <think> </think> tags and then output the final answer in <answer> </answer> tags."


Preprocessing dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

In [14]:
def add_reasoning_output(item):
    conversation = make_conversation_from_prompt(item, reasoning_prompt_template)

    text = processor.apply_chat_template(
        conversation['messages'], tokenize=False, add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(conversation['messages'])

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(reasoning_model.device)

    # Inference: Generation of the output
    with torch.no_grad():
        # Generate the output
        generated_ids = reasoning_model.generate(**inputs, max_new_tokens=256)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text_batch = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        assert len(output_text_batch) == 1
        output_text = output_text_batch[0]
    return {
        "desired_output": output_text,
    }
dataset = dataset.map(add_reasoning_output)

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

In [16]:
for key, value in dataset[0].items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {value.shape}")
    else:
        print(f"{key}: {value}")
    print(20 * "-")

image_path: /scratch/izar/vanousek/vlm_r1/data/images/vsr/000000558388.jpg
--------------------
problem: Is the following statement true: The cake is next to the person.
--------------------
solution: True
--------------------
desired_output: <think>
The image shows a close-up of a chocolate cake with candles and colorful candies on top, placed on a table. There is no visible person in the frame, so it cannot be determined if the cake is next to someone.
</think>
<answer>
True
</answer>
--------------------


In [17]:
dataset.save_to_disk(OUTPUT_DATASET_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/35 [00:00<?, ? examples/s]